In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
USER_IDS = [46343, 759667, 781756, 844359, 1066528, 1360686, 1449548, 1455390, 1818471, 2598705, 2638030, 3509524, 3997827, 4018081, 4314139, 4426783, 5132496, 5383425, 5498603, 5797046, 6220552, 7749105, 8000685, 8173033, 8258170, 8530312, 8686948, 8692923, 9106476, 9618981, 9961348]


In [3]:
import os
import pandas as pd
import numpy as np

ALPHA = 0.05
EXCLUDE_EPOCHS = []

LABELS_DIR = "../data/lables/"
PROCESSED_DIR = "../data/processed/"
FINAL_DIR = "../data/dataset_ml/"
os.makedirs(FINAL_DIR, exist_ok=True)

# розрахунок RMSSD на сирих RR-інтервалах
def calc_rmssd(rr_array):
    if len(rr_array) > 1:
        diffs = np.diff(rr_array)
        return np.sqrt(np.mean(diffs**2))
    return 0

def calc_rr_delta(rr_array):
    if len(rr_array) > 0:
        return np.max(rr_array) - np.min(rr_array)
    return 0


for uid in USER_IDS:
    print(f"[{uid}] Збірка фінального датасету...")
    try:
        labels_file = f"{LABELS_DIR}{uid}_labeled_sleep.txt"
        hr_file = f"{PROCESSED_DIR}{uid}_clean_hr.csv"
        motion_file = f"{PROCESSED_DIR}{uid}_clean_motion.csv"

        if not (os.path.exists(labels_file) and os.path.exists(hr_file) and os.path.exists(motion_file)):
            print(f"  -> Пропуск: не вистачає вихідних файлів для {uid}")
            continue

        # Мітки
        labels_df = pd.read_csv(labels_file, sep=r'\s+', header=None, names=['time', 'sleep_stage'])
        global_t_start = labels_df['time'].min()
        labels_df['epoch_id'] = ((labels_df['time'] - global_t_start) // 30).astype(int)

        # Пульс
        hr_df = pd.read_csv(hr_file)
        hr_df['epoch_id'] = ((hr_df['time'] - global_t_start) // 30).astype(int)
        hr_df = hr_df[hr_df['epoch_id'] >= 0]


        #  'rr_norm_diff' залишаємо, щоб не ламати код
        hr_features = hr_df.groupby('epoch_id').agg(
            mean_rr=('rr', 'mean'),
            std_rr=('rr', 'std'),
            rr_norm_diff=('rr', calc_rmssd),
            rr_delta=('rr', calc_rr_delta)
        ).reset_index()

        # ====================================================================
        # СТАРИЙ ПІДХІД
        # ====================================================================
        # hr_features = hr_df.groupby('epoch_id').agg(
        #     mean_rr=('rr', 'mean'),
        #     std_rr=('rr', 'std')
        # ).reset_index()
        #
        # ema_baseline = np.zeros(len(hr_features))
        # ema_baseline[0] = hr_features['mean_rr'].iloc[0]
        # for i in range(1, len(hr_features)):
        #     ema_baseline[i] = ALPHA * hr_features['mean_rr'].iloc[i] + (1 - ALPHA) * ema_baseline[i-1]
        # hr_features['rr_norm_diff'] = hr_features['mean_rr'] - ema_baseline
        # hr_features['rr_delta'] = hr_features['mean_rr'].diff().fillna(0)
        # ====================================================================

        # Рух
        motion_df = pd.read_csv(motion_file)
        motion_df['epoch_id'] = ((motion_df['time'] - global_t_start) // 30).astype(int)
        motion_df = motion_df[motion_df['epoch_id'] >= 0]

        motion_features = motion_df.groupby('epoch_id').agg(
            mean_enmo=('enmo', 'mean'),
            max_enmo=('enmo', 'max'),
            std_enmo=('enmo', 'std')
        ).reset_index()

        # ЗЛИТТЯ
        final_df = labels_df[['epoch_id', 'sleep_stage']].merge(hr_features, on='epoch_id', how='inner')
        final_df = final_df.merge(motion_features, on='epoch_id', how='inner')

        # - пусті епохи
        initial_len = len(final_df)
        final_df = final_df.dropna()
        dropped_rows = initial_len - len(final_df)
        if dropped_rows > 0:
            print(f"  -> Увага: видалено {dropped_rows} епох через відсутність пульсу/руху.")

        for start_ep, end_ep in EXCLUDE_EPOCHS:
            final_df = final_df[~((final_df['epoch_id'] >= start_ep) & (final_df['epoch_id'] <= end_ep))]

        # Збереження
        out_path = f"{FINAL_DIR}{uid}_30_dataset.csv"
        final_df.to_csv(out_path, index=False)
        print(f"  -> ГОТОВО: {len(final_df)} ідеальних епох збережено в {out_path}\n")

    except Exception as e:
        print(f"  -> ПОМИЛКА у юзера {uid}: {e}\n")

[46343] Збірка фінального датасету...
  -> Увага: видалено 1 епох через відсутність пульсу/руху.
  -> ГОТОВО: 566 ідеальних епох збережено в ../data/dataset_ml/46343_30_dataset.csv

[759667] Збірка фінального датасету...
  -> ГОТОВО: 454 ідеальних епох збережено в ../data/dataset_ml/759667_30_dataset.csv

[781756] Збірка фінального датасету...
  -> ГОТОВО: 979 ідеальних епох збережено в ../data/dataset_ml/781756_30_dataset.csv

[844359] Збірка фінального датасету...
  -> ГОТОВО: 947 ідеальних епох збережено в ../data/dataset_ml/844359_30_dataset.csv

[1066528] Збірка фінального датасету...
  -> Увага: видалено 2 епох через відсутність пульсу/руху.
  -> ГОТОВО: 816 ідеальних епох збережено в ../data/dataset_ml/1066528_30_dataset.csv

[1360686] Збірка фінального датасету...
  -> ГОТОВО: 927 ідеальних епох збережено в ../data/dataset_ml/1360686_30_dataset.csv

[1449548] Збірка фінального датасету...
  -> ГОТОВО: 898 ідеальних епох збережено в ../data/dataset_ml/1449548_30_dataset.csv

[14